In [ ]:
lab.gocator.disconnect()

In [ ]:
lab.disconnect_all()

In [ ]:
%load_ext autoreload
%autoreload 2

# Hardware check — Gocator realignment, then the tile-seam fix (#58, PR #59)

Two parts:

1. **Realign the Gocator** after a remount. Solves `gocator.mounting` and
   `frames.instruments.gocator.translation` against the **WTT12L dot**, which is the gantry's
   reference point (its frame translation is `[0, 0, 0]`). You paste the result into the config.
2. **Seam test.** Runs the same 2-pass serpentine tile twice: once with the old behaviour, once with
   the fix. Then it measures how far the forward and reverse swaths disagree along travel.

| § | What | Moves? |
|---|---|---|
| 0 | Parameters, helpers, connect in safe_mode | no |
| 1a | Stationary Gocator profile: is the bed inside the active area? | no |
| 1b | Enable motion — **explicit**; STOP cell | — |
| 1c | WTT12L: put the dot on the block, then edge transects → block centre in gantry mm | **yes** |
| 1d | Three Gocator passes over the block (fwd, fwd stepped in Y, reverse) | **yes** |
| 1e | Mounting check: which axis map makes the three passes agree | no |
| 1f | Solve the translation; print the YAML to paste | no |
| 1g | Reload from the edited config, confirm it took | no |
| 2a | Plan the tile around the block; dry-run targets vs soft limits and fences | no |
| 2b | Re-enable motion; run A (old behaviour) and run B (fix) | **yes** |
| 2c | Seam analysis and plots | no |
| 2d | Summary, park, safe_mode back on | **yes** (park) |

## Reusing earlier results

Every result is saved to the run's folder as it appears (block position, passes, picked corners,
solution). Set `LOAD_FROM` (a folder, or `"latest"`) or `B_W_KNOWN` in the parameters cell and §1c / §1d
skip the scans you already have. Loading never moves anything. See "Reuse saved results" under §0.

## What you need

- A **rectangular, non-square** block, e.g. 100 × 200 mm and 30–80 mm tall, with a matte top.
  Square it up so its **long side runs along gantry X**. Measure it and set `BLOCK_X_MM` and
  `BLOCK_Y_MM`. Non-square matters: it is how a swapped axis pairing shows up.
- Put it on clear floor, away from fences and from the ends of travel. Part 2's survey (`SURVEY_X_MM`, `SURVEY_Y_MM`)
  needs its X span of travel plus the lead-ins, and its Y span inside the Y travel.
- Leave it in place for the whole notebook. Every later step uses its position from §1c.

## How to stop motion — read before §1b

1. **The STOP cell** (§1b, it only contains `lab.pause()`). Moves don't block the kernel, so you
   can run it any time. Keep it on screen.
2. **A cell that is still running** (`[*]`, e.g. a scan pass): press **Interrupt** (⏹ / `I`,`I`),
   then run the STOP cell.
3. **From a terminal** in the repo root: `touch ESTOP` (hard halt) or `touch PAUSE` (resumable).
   §0 starts the sentinel watcher. Delete the file before `rearm()` / `resume()`.
4. **Physical e-stop** — always the last word.

block is 7 3/4 in (196.85mm) long and 3 7/8 (98.425) in wide and 1 3/4 tall (44.45)

## 0. Parameters, helpers, connect (no motion)

In [ ]:
import dataclasses, json, logging, os, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyprojroot import here
os.chdir(here())

from laguna import FlumeLab
from laguna.safety import SafetyState
from laguna.frames import AffineTransform, FrameRegistry, InstrumentFrame, orient_scan
from laguna.robot.macron.commands import ramp_distance_mm, ramp_time_s
from laguna.scanner.mounting import SensorMounting
from laguna.survey import SurveyRunner, Tile
from laguna.alignment_store import AlignmentStore

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

# ---- Parameters: check these against the rig before §1b ----
CONFIG = "experiments/delta-experiment-basin-C/config.yaml"
SIMULATE = False            # True rehearses every cell with no hardware (readings are NaN/synthetic)

# The block — measure it (mm)
BLOCK_X_MM = 196.85          # side along gantry X
BLOCK_Y_MM = 98.425         # side along gantry Y
BLOCK_MIN_HEIGHT_MM = 20.0  # "above the bed" threshold; keep it well under the block's real height
BLOCK_GUESS_XY = (1000.0, 600.0)  # gantry X, Y that puts the WTT12L dot roughly on the block centre (§1c)

# Speeds (mm/s)
SCAN_SPEED = 100.0          # Gocator passes; matching gocator.travel_speed_mm_s avoids a flash write
TRAVEL_SPEED = 100.0        # repositioning
WTT_SPEED = 10.0            # WTT12L transects: slow keeps the edges sharp

# Geometry (mm)
WTT_MARGIN_MM = 60.0        # WTT12L transects run this far past each block edge
ALIGN_PASS_LEN_MM = 600.0   # §1d pass length along X, centred on the block
ALIGN_STEP_Y_MM = None      # §1d P1 -> P2 step along Y; None = 0.4 x active-area width
# §2 survey boundaries, experiment frame, mm. None = centred on the block.
SURVEY_X_MM = None          # (x_min, x_max) along travel; None = TILE_LEN_MM centred on the block
SURVEY_Y_MM = None          # (y_min, y_max) across; passes + overlap are SOLVED to cover this; narrower than one swath = a single pass
TILE_LEN_MM = 600.0         # used only when SURVEY_X_MM is None
MIN_OVERLAP = 0.10          # fewest passes that keep at least this swath overlap (spread evenly)
SCAN_GANTRY_AXIS = "X"     # gantry axis every pass scans along, whatever the experiment frame's rotation

# Pass/fail
SEAM_TOL_MM = 2.0           # run B forward-vs-reverse travel offset must be within this
RUN_BASELINE = True         # §2b also runs the old behaviour (run A) for comparison

# Reuse earlier results instead of re-scanning (no motion either way). Only valid while the block hasn't moved.
LOAD_FROM = "latest"            # None | "latest" | path of a previous seam_test_* folder
B_W_KNOWN = None            # or type the block centre you already know: (gantry X, gantry Y) mm; skips the WTT12L scan

ALLOW_MOTION = False        # §1b and §2b refuse to enable motion unless this is True

OUT = Path("data/scans") / time.strftime("seam_test_%Y%m%d_%H%M%S")
OUT.mkdir(parents=True, exist_ok=True)
print("saving to", OUT)

In [ ]:
# ---- Helpers ----
results = []
def check(name, ok, detail=""):
    results.append((name, bool(ok), detail))
    print(("PASS " if ok else "FAIL ") + name + (f" — {detail}" if detail else ""))

def pos(axis=None):
    p = dict(zip([a.name for a in lab.gantry._axes], lab.gantry.get_position()))
    return p if axis is None else p[axis]

def vec(**over):
    # Full per-axis position vector (acquire_scan's form): current position with `over` applied.
    p = pos(); p.update(over)
    return [p[a.name] for a in lab.gantry._axes]

# Identity experiment frame and zero Gocator offset: orient_scan() with this gives
# "gantry position + mounting-rotated sensor point", i.e. block positions in gantry mm
# minus the (unknown) Gocator translation.
GANTRY_ONLY = FrameRegistry()

def scan_points(scan, frames=GANTRY_ONLY, mounting=None):
    # Valid (N, 3) points of an oriented scan, optionally re-oriented with a candidate mounting.
    if mounting is not None:
        scan = dataclasses.replace(scan, mounting=mounting)
    return orient_scan(scan, frames=frames).to_points(drop_invalid=True)

def find_block(points, min_height=None):
    # The block's top: points at least `min_height` above the bed (the swath's 20th
    # percentile height, since the bed dominates a swath). Footprint edges are the 1st/99th
    # percentiles of the top points, which ignores stray returns. None if no block is seen.
    min_height = BLOCK_MIN_HEIGHT_MM if min_height is None else min_height
    if points is None or len(points) < 100:
        return None
    z = points[:, 2]
    bed = float(np.nanpercentile(z, 20))
    top = points[z > bed + min_height]
    if len(top) < 50:
        return None
    lo = np.percentile(top[:, :2], 1, axis=0)
    hi = np.percentile(top[:, :2], 99, axis=0)
    return {"center": (lo + hi) / 2, "size": hi - lo, "lo": lo, "hi": hi,
            "bed_z": bed, "top_z": float(np.median(top[:, 2])), "n": int(len(top))}

def transect_edges(pos_mm, signal, merge_mm=3.0):
    # Where `signal` crosses midway between its low and high plateaus, by linear interpolation.
    # Crossings closer than `merge_mm` (noise right at an edge) are merged into one.
    ok = np.isfinite(pos_mm) & np.isfinite(signal)
    pos_mm, signal = pos_mm[ok], signal[ok]
    order = np.argsort(pos_mm)
    pos_mm, signal = pos_mm[order], signal[order]
    if len(pos_mm) < 10:
        return []
    lo, hi = np.percentile(signal, [10, 90])
    mid = (lo + hi) / 2
    above = signal > mid
    idx = np.nonzero(above[1:] != above[:-1])[0]
    xs = [pos_mm[i] + (mid - signal[i]) * (pos_mm[i + 1] - pos_mm[i]) / (signal[i + 1] - signal[i])
          for i in idx]
    merged = []
    for x in xs:
        if merged and x - merged[-1][-1] < merge_mm:
            merged[-1].append(x)
        else:
            merged.append([x])
    return [float(np.mean(g)) for g in merged]

def block_from_transect(pos_mm, signal):
    # Centre and width of the block along one transect, from its two edges.
    e = transect_edges(pos_mm, signal)
    if len(e) != 2:
        return None, e
    return {"center": (e[0] + e[1]) / 2, "width": e[1] - e[0]}, e

def grid_heights(points, x_edges, y_edges):
    s, _, _ = np.histogram2d(points[:, 0], points[:, 1], bins=[x_edges, y_edges], weights=points[:, 2])
    n, _, _ = np.histogram2d(points[:, 0], points[:, 1], bins=[x_edges, y_edges])
    with np.errstate(invalid="ignore", divide="ignore"):
        return s / n

def travel_shift(pa, pb, y_range, x_range, cell=1.0, max_shift=80.0):
    # Shift along X (mm) that best maps pass a's heights onto pass b's in a strip both cover:
    # positive means features appear further +X in pass b. Normalized correlation of the
    # mean-removed height grids, scanned over integer-cell shifts. Returns (shift, peak corr).
    xe = np.arange(x_range[0], x_range[1] + cell, cell)
    ye = np.arange(y_range[0], y_range[1] + 5 * cell, 5 * cell)
    if len(xe) < 10 or len(ye) < 2:
        return float("nan"), float("nan")
    A = grid_heights(pa, xe, ye); A -= np.nanmean(A)
    B = grid_heights(pb, xe, ye); B -= np.nanmean(B)
    best = (float("nan"), -np.inf)
    k = int(max_shift / cell)
    for d in range(-k, k + 1):
        a = A[max(0, -d):A.shape[0] - max(0, d)]
        b = B[max(0, d):B.shape[0] - max(0, -d)]
        m = np.isfinite(a) & np.isfinite(b)
        if m.sum() < 100:
            continue
        r = float(np.sum(a[m] * b[m]) / np.sqrt(np.sum(a[m] ** 2) * np.sum(b[m] ** 2)))
        if r > best[1]:
            best = (d * cell, r)
    return best

def travel_at(t, v, a):
    # Distance covered t seconds after starting from rest: trapezoidal ramp up to v at accel a.
    if not a or a <= 0:
        return v * t
    tr = v / a
    return 0.5 * a * t * t if t <= tr else 0.5 * v * tr + v * (t - tr)

def candidate_mountings(scan_z):
    # Every proper rotation that keeps the configured scan_z (the sensor still looks down):
    # scan_x on one of +-X / +-Y, scan_y on the other horizontal axis with the sign the
    # determinant forces. Mirrors are rejected by SensorMounting itself.
    out = []
    for sx in ("+X", "-X", "+Y", "-Y"):
        other = "Y" if sx[1] == "X" else "X"
        for sy in ("+" + other, "-" + other):
            try:
                out.append(SensorMounting(scan_x=sx, scan_y=sy, scan_z=scan_z))
            except ValueError:
                pass
    return out

def score_mounting(m, p1, p2, p3):
    # How well mounting `m` makes the three §1d passes agree with each other and with the
    # block's real size. Lower is better; inf if the block isn't found in every pass.
    blocks = [find_block(scan_points(s, mounting=m)) for s in (p1, p2, p3)]
    if any(b is None for b in blocks):
        return float("inf"), blocks, {}
    b1, b2, b3 = blocks
    size_err = float(np.max(np.abs(b1["size"] - np.array([BLOCK_X_MM, BLOCK_Y_MM]))))
    dy12 = float(b2["center"][1] - b1["center"][1])
    dx13 = float(b3["center"][0] - b1["center"][0])
    return size_err + abs(dy12) + abs(dx13), blocks, {"size_err": size_err, "dy12": dy12, "dx13": dx13}

In [ ]:
lab = FlumeLab(CONFIG, simulate=SIMULATE)
lab.add("gantry")
lab.add("gocator")
assert SIMULATE or lab.gantry._safe_mode is True, "config should start in safe_mode"   # the simulator turns it off
check("connect_all", lab.connect_all())
lab.watch_for_safety()   # ./PAUSE, ./STOP, ./ESTOP sentinel files
START = pos()
print("start position:", START)

print("\nconfigured gocator.mounting:", lab.gocator.mounting.to_dict())
print("configured frames:", json.dumps(lab.frames.describe(), indent=1, default=str))
AREA = lab.gocator.get_active_area()
print("active area:", {k: round(v, 1) for k, v in AREA.items()})
ACCEL_X = lab.gantry.axis("X").get_accel()
print(f"X accel {ACCEL_X:.1f} mm/s² -> at {SCAN_SPEED} mm/s: ramp {ramp_distance_mm(SCAN_SPEED, ACCEL_X):.1f} mm, "
      f"{ramp_time_s(SCAN_SPEED, ACCEL_X):.3f} s")

### Reuse saved results (no motion)

Each result of this notebook is written to this run's folder (`OUT`) the moment it exists: the
block position from the WTT12L scan (`block_location.json`), the three passes (`align_P*.npz`), the
picked corners (`block_corners.json`) and the solution (`alignment.json`). To skip a scan you've already
done, set `LOAD_FROM` in the parameters cell (a folder, or `"latest"` for the newest run that has the
result), or type the block centre as `B_W_KNOWN`, then run the next cell. Everything it loads is
reported, and §1c and §1d skip their scans for whatever was loaded.

Only reuse the block position while the block hasn't moved, and only reuse passes with the config they
were aimed with. Saved corner picks go with the saved passes they were made on.

In [ ]:
store = AlignmentStore(OUT)     # this run's results land here as they are produced
B_W, HAVE_BLOCK, HAVE_PASSES, LOADED_CORNERS, reused, first_src = None, False, False, {}, [], None

def _source(having):
    if LOAD_FROM is None:
        return None
    if LOAD_FROM == "latest":
        return AlignmentStore.latest(OUT.parent, having=having)
    return AlignmentStore(LOAD_FROM)

if B_W_KNOWN is not None:
    B_W, HAVE_BLOCK = np.asarray(B_W_KNOWN, dtype=float), True
    reused.append(f"block centre typed in: {B_W.round(1).tolist()}")

src = _source("passes")
if src is not None:
    ps = src.load_passes()
    if len(ps) == 3:
        P1, P2, P3 = ps["P1"], ps["P2"], ps["P3"]
        HAVE_PASSES, LOADED_CORNERS, first_src = True, src.load_corners(), src
        reused.append(f"three passes <- {src.directory.name}" + (f" (+{len(LOADED_CORNERS)} corner picks)" if LOADED_CORNERS else ""))
        if B_W is None and src.load_block() is not None:     # the passes were aimed at this block
            B_W, HAVE_BLOCK = src.load_block(), True
            reused.append(f"block centre <- {src.directory.name}: {B_W.round(1).tolist()}")

if B_W is None:
    src = _source("block")
    if src is not None and src.load_block() is not None:
        B_W, HAVE_BLOCK, first_src = src.load_block(), True, first_src or src
        reused.append(f"block centre <- {src.directory.name}: {B_W.round(1).tolist()}")

if reused:
    print("reusing:\n  " + "\n  ".join(reused))
    if first_src is not None:
        store.note_loaded_from(first_src, reused)
    print("skipping:", ", ".join(n for n, h in (("§1c WTT12L scan", HAVE_BLOCK), ("§1d Gocator passes", HAVE_PASSES)) if h) or "nothing")
else:
    print("nothing loaded: every scan will run")


## 1a. Stationary profile — is the bed inside the active area? (no motion)

One laser line, no motion. After a remount the standoff can change. If the bed or the block top
falls outside the active area's Z window (dashed lines), fix `gocator.active_area` (`z`, `height`)
in the config first, then restart the kernel. The dotted lines are the X window: the width one swath covers, which Part 2 uses as `swath_mm`.
The profile is in the sensor's *aligned* frame (web UI alignment) and the window is applied in that
same frame, so the two are compared directly. Anything under the line will do; the block doesn't
have to be.


In [ ]:
prof = lab.gocator.scan_profile()
sp = prof.to_points(frame="sensor", drop_invalid=False)
valid = np.isfinite(sp[:, 2])
z_lo, z_hi = AREA["z_mm"], AREA["z_mm"] + AREA["height_mm"]
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(sp[valid, 0], sp[valid, 2], ".", ms=1)
for z in (z_lo, z_hi):
    ax.axhline(z, ls="--", c="gray")
for x in (AREA["x_mm"], AREA["x_mm"] + AREA["width_mm"]):
    ax.axvline(x, ls=":", c="gray")
ax.set_xlabel("X (across the laser), mm"); ax.set_ylabel("Z (range), mm")
ax.set_title("stationary profile"); plt.show()
frac = float(valid.mean()) if len(valid) else 0.0
check("most of the line returns", frac > 0.5, f"{frac:.0%} valid")
if valid.any():
    zv = sp[valid, 2]
    margin = min(np.percentile(zv, 1) - z_lo, z_hi - np.percentile(zv, 99))
    check("returns sit inside the Z window with margin", margin > 5.0, f"closest {margin:.1f} mm from a window edge")

# configure() otherwise stays in the last mode used; the scans below need surface mode.
lab.gocator.configure(mode="surface")


## 1b. Enable motion — explicit

Everything below moves the gantry. Re-read "How to stop motion" at the top.

The next cell is the **STOP cell**. It only pauses; run it any time. Then re-run the cell *after* it
to enable motion again.

In [ ]:
# ===================== STOP CELL =====================
lab.pause()          # cancels any move in flight; lab.resume() to continue

In [ ]:
if not ALLOW_MOTION:
    raise SystemExit("ALLOW_MOTION is False — set it True in the parameters cell once the rig is clear.")
if lab.safety_state is not SafetyState.RUNNING:
    lab.resume()
check("set_safe_mode(False)", lab.gantry.set_safe_mode(False))

## 1c. WTT12L: block centre in gantry coordinates

**Step 1.** Run the next cell. It moves to `BLOCK_GUESS_XY`. Look at the WTT12L's red dot. If it
isn't on the block top near the centre, edit `BLOCK_GUESS_XY` in the parameters cell, re-run that
cell, and re-run this one. Within about 20 mm of the centre is plenty.

**Step 2.** The cell after it runs four slow transects across the block: X out and back, then Y
out and back. Each transect finds both block edges where `current_ma` crosses halfway between the
bed and block-top levels. Averaging the out and back directions cancels the lag between a sample
and its recorded position. The result is `B_W`, the block centre in gantry X/Y. Every later step
uses it.

In [ ]:
BLOCK_GUESS_XY = (1350, 650)

In [ ]:
# lab.gocator.start()

In [ ]:
# lab.move_to(X=1040, Y=560, speed=TRAVEL_SPEED).wait()

In [ ]:
# lab.move_to(X=0, Y=0, speed=TRAVEL_SPEED).wait()

In [ ]:
if HAVE_BLOCK:
    print("block centre already known: skipping the move to the guess")
else:
    lab.move_to(X=BLOCK_GUESS_XY[0], Y=BLOCK_GUESS_XY[1], speed=TRAVEL_SPEED).wait()
    print("Is the WTT12L dot on the block top, near its centre? If not, adjust BLOCK_GUESS_XY and re-run.")

In [ ]:
bx, by = BLOCK_GUESS_XY
plan = [("X", bx - BLOCK_X_MM / 2 - WTT_MARGIN_MM, bx + BLOCK_X_MM / 2 + WTT_MARGIN_MM, {"Y": by}),
        ("Y", by - BLOCK_Y_MM / 2 - WTT_MARGIN_MM, by + BLOCK_Y_MM / 2 + WTT_MARGIN_MM, {"X": bx})]
if HAVE_BLOCK:
    print(f"block centre already known {B_W.round(1).tolist()}: skipping the WTT12L scan")
else:
    wtt = {}
    fig, axes = plt.subplots(1, 2, figsize=(11, 3))
    for (axis, lo, hi, fixed), ax in zip(plan, axes):
        found = []
        for a, b in ((lo, hi), (hi, lo)):
            res = lab.acquire_scan("wtt12l", start=vec(**fixed, **{axis: a}), end=vec(**fixed, **{axis: b}),
                                   feed_rate_mm_s=WTT_SPEED, axis=axis)
            df = pd.read_csv(res.path)
            blk, edges = block_from_transect(df["pos_mm"].to_numpy(float), df["current_ma"].to_numpy(float))
            ax.plot(df["pos_mm"], df["current_ma"], ".", ms=2, label=f"{a:.0f} → {b:.0f}")
            for e in edges:
                ax.axvline(e, c="gray", lw=0.5)
            print(f"{axis} {a:.0f} -> {b:.0f}: edges {[round(e, 1) for e in edges]}  ({res.path})")
            found.append(blk)
        ax.set_xlabel(f"gantry {axis}, mm"); ax.set_ylabel("current_ma"); ax.legend()
        if all(found):
            wtt[axis] = {"center": (found[0]["center"] + found[1]["center"]) / 2,
                         "width": (found[0]["width"] + found[1]["width"]) / 2,
                         "lag": (found[0]["center"] - found[1]["center"]) / 2}
    plt.show()

    if set(wtt) == {"X", "Y"}:
        B_W = np.array([wtt["X"]["center"], wtt["Y"]["center"]])
        print(f"block centre B_W = ({B_W[0]:.1f}, {B_W[1]:.1f}) gantry mm")
        store.save_block(B_W, wtt, (BLOCK_X_MM, BLOCK_Y_MM))
        for axis, real in (("X", BLOCK_X_MM), ("Y", BLOCK_Y_MM)):
            w = wtt[axis]
            check(f"WTT12L {axis} width matches the block", abs(w["width"] - real) < 6.0,
                  f"{w['width']:.1f} mm vs {real} mm; out/back half-difference {w['lag']:+.1f} mm")
    else:
        B_W = None
        check("WTT12L found both edges in every transect", False,
              "need exactly two edges per transect — is the dot on the block, and the margin wide enough?")

In [ ]:
B_W

## 1d. Three Gocator passes over the block

All three use the fixed path: move to a ramp start, then `scan_with_gantry(..., cruise_start_mm=...)`.
The block lands on one side of the fan in P1 and on the other side in P2. P3 is P1 run in reverse.

The passes are aimed with the Gocator offset **from the current config**. If the remount moved the
sensor by more than about `0.2 × fan width` across, or `ALIGN_PASS_LEN_MM / 2 - 100` along X, the
block may be missed. If so, widen `ALIGN_PASS_LEN_MM` or set `GOC_GUESS` below by hand (gantry mm
from the WTT12L dot to the Gocator).

In [ ]:
lab.gantry.get_position()

In [ ]:
assert B_W is not None, "§1c must find the block first"
GOC_GUESS = lab.frames.frame_for("gocator").offset[:2]   # override with your own rough (dx, dy) if needed
GOC_GUESS = [597, 312]
W = AREA["width_mm"]
STEP_Y = ALIGN_STEP_Y_MM or 0.4 * W
cx, cy = B_W - GOC_GUESS                 # gantry X/Y that puts the (guessed) Gocator on the block
xa, xb = cx - ALIGN_PASS_LEN_MM / 2, cx + ALIGN_PASS_LEN_MM / 2
y1, y2 = cy - STEP_Y / 2, cy + STEP_Y / 2

In [ ]:
cy

In [ ]:
def gocator_pass(y, x_from, x_to, speed=SCAN_SPEED):
    # One pass along X in gantry mm, through the fixed path: lead-in, then cruise_start_mm.
    accel = lab.gantry.axis("X").get_accel()
    lead = ramp_distance_mm(speed, accel) * (1 if x_to > x_from else -1)
    lab.move_to(X=x_from - lead, Y=y, speed=TRAVEL_SPEED).wait()
    return lab.gocator.scan_with_gantry(
        lab.gantry, axis="X", end_mm=x_to, feed_rate_mm_s=speed,
        settle_s=ramp_time_s(speed, accel), cruise_start_mm=x_from,
    )

if HAVE_PASSES:
    print("passes loaded: skipping the three Gocator passes")
else:
    print(f"P1/P3 at Y={y1:.0f}, P2 at Y={y2:.0f}; X {xa:.0f} <-> {xb:.0f}")
    P1 = gocator_pass(y1, xa, xb)
    P2 = gocator_pass(y2, xa, xb)
    P3 = gocator_pass(y1, xb, xa)
    for name, s in (("P1", P1), ("P2", P2), ("P3", P3)):
        store.save_pass(name, s)
        print(name, "valid points:", s.valid_count)

## 1e. Mounting check (no motion)

The captured scans are re-oriented with every axis map that keeps the configured `scan_z` (the
sensor still faces down). Each candidate is scored on three things:

- **Footprint vs the block's real size.** A swapped axis pairing shows up as a transposed
  footprint, or a travel-scale error as a stretched one.
- **P1 vs P2 along Y.** With the wrong `scan_x` sign, the block moves by about 2 × `STEP_Y`
  between them.
- **P1 vs P3 along X.** `orient_scan()` takes the along-travel direction from each pass's
  start→end alone, so `scan_y`'s sign doesn't move the scans. Forward and reverse passes should
  land on the same stretch of X. A few mm to about 10 mm of disagreement is the seam residual
  Part 2 measures (start latency, opposite sign in each direction), not a mounting error. Tens to
  hundreds of mm means the direction handling or the passes' recorded start/end is wrong.

If no candidate scores well, check the passes' ranges first (`X` range of each oriented pass
should sit inside its start→end), then `docs/subsystems/frames.md`, "FYI: the Gocator has a
source-level mirror setting".

In [ ]:
mnt = lab.gocator.mounting
fig, axes = plt.subplots(3, 3, figsize=(16, 11))
for col, (name, s) in enumerate((("P1 fwd", P1), ("P2 fwd, stepped", P2), ("P3 reverse", P3))):
    p = scan_points(s, mounting=mnt)
    z = p[:, 2]
    bed = float(np.nanpercentile(z, 20))
    thr = bed + BLOCK_MIN_HEIGHT_MM
    b = find_block(p)
    print(f"{name}: {len(p)} valid pts | X {p[:,0].min():.0f}..{p[:,0].max():.0f}  "
          f"Y {p[:,1].min():.0f}..{p[:,1].max():.0f}  Z {z.min():.1f}..{z.max():.1f} | "
          f"bed(p20)={bed:.1f}  thr={thr:.1f}  above thr: {(z > thr).sum()} | "
          f"block: {None if b is None else (np.round(b['center'], 1).tolist(), np.round(b['size'], 1).tolist())}")

    # 1. top-down map, colour clipped to the interesting height range
    q = p[::max(1, len(p) // 60000)]
    ax = axes[0, col]
    sc = ax.scatter(q[:, 0], q[:, 1], c=q[:, 2], s=1, vmin=np.percentile(z, 1), vmax=np.percentile(z, 99.5))
    plt.colorbar(sc, ax=ax, label="Z, mm")
    if b is not None:
        ax.add_patch(plt.Rectangle(b["lo"], *b["size"], fill=False, ec="r"))
    ax.set_title(name); ax.set_xlabel("X, mm"); ax.set_ylabel("Y, mm"); ax.set_aspect("equal")

    # 2. height histogram with the detection thresholds
    ax = axes[1, col]
    ax.hist(z, bins=200, log=True)
    ax.axvline(bed, c="g", label=f"bed p20 {bed:.1f}")
    ax.axvline(thr, c="r", ls="--", label=f"threshold {thr:.1f}")
    ax.set_xlabel("Z, mm"); ax.legend()

    # 3. Z vs the across-fan axis (the one with the widest spread other than travel), all points
    ax = axes[2, col]
    across = 1 if np.ptp(p[:, 1]) > np.ptp(p[:, 0]) else 0
    ax.plot(p[::max(1, len(p) // 40000), across], q[:0, 2].tolist() or p[::max(1, len(p) // 40000), 2], ".", ms=1)
    ax.axhline(thr, c="r", ls="--")
    ax.set_xlabel("XY"[across] + ", mm"); ax.set_ylabel("Z, mm")
plt.tight_layout(); plt.show()

In [ ]:
bed

### Pick the block (interactive)

Needs `ipympl` in the kernel's environment (`pip install ipympl`) and a restart of the kernel after
installing. Run the next cell: it switches to the widget backend and shows P1–P3 in the same frame
the passes were aimed in. Use the toolbar to zoom, and **with no toolbar tool active** click the
block's **four corners** on each map (a 5th click starts over). The rectangle you get is only a
region of interest: the block's edges and centre still come from the heights inside it. Then run
the cell after that one to apply it.


In [ ]:
%matplotlib widget
import ipympl  # noqa: F401  (fails here, clearly, if it isn't installed)

CLICKS = {"P1": [], "P2": [], "P3": []}
_art = {}

def _on_click(event, name, ax):
    if event.inaxes is not ax or event.xdata is None:
        return
    if getattr(event.canvas, "toolbar", None) is not None and event.canvas.toolbar.mode:
        return                                   # zoom/pan tool is active: not a pick
    pts = CLICKS[name]
    if len(pts) >= 4:                            # start over
        pts.clear()
        for art in _art.pop(name, []):
            art.remove()
    pts.append((event.xdata, event.ydata))
    _art.setdefault(name, []).extend(ax.plot(event.xdata, event.ydata, "r+", ms=14))
    if len(pts) == 4:
        a = np.array(pts)
        x0, y0 = a.min(axis=0); x1, y1 = a.max(axis=0)
        _art[name].append(ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="r")))
        ax.set_title(f"{name}: {x1 - x0:.0f} x {y1 - y0:.0f} mm  (block is {BLOCK_X_MM:.0f} x {BLOCK_Y_MM:.0f})")
    event.canvas.draw_idle()

pick_fig, pick_axes = plt.subplots(1, 3, figsize=(17, 6))
for ax, (name, s) in zip(pick_axes, (("P1", P1), ("P2", P2), ("P3", P3))):
    p = scan_points(s, mounting=lab.gocator.mounting)
    q = p[::max(1, len(p) // 60000)]
    sc = ax.scatter(q[:, 0], q[:, 1], c=q[:, 2], s=1,
                    vmin=np.percentile(p[:, 2], 1), vmax=np.percentile(p[:, 2], 99.5))
    ax.set_aspect("equal"); ax.set_title(f"{name}: click the 4 block corners")
    ax.set_xlabel("X, mm"); ax.set_ylabel("Y, mm")
    pick_fig.canvas.mpl_connect("button_press_event", lambda e, n=name, a=ax: _on_click(e, n, a))
plt.tight_layout(); plt.show()


In [ ]:
%matplotlib inline
import json

MARGIN_MM = 15.0                      # ROI = picked corners' bounding box, grown by this
MANUAL = dict(LOADED_CORNERS)         # loaded with the passes; or type {"P1": (x0, y0, x1, y1), ...} to skip clicking
# MANUAL = CLICKS

# Wrap once, even if this cell is re-run.
_orig_scan_points = globals().get("_orig_scan_points", scan_points)
_orig_find_block = globals().get("_orig_find_block", find_block)
ROI_MASK = {}                         # id(scan) -> bool mask over that scan's valid points

def scan_points(scan, frames=GANTRY_ONLY, mounting=None):
    # Same as before, plus a 4th column: 1.0 where the point is inside the picked ROI.
    p = _orig_scan_points(scan, frames=frames, mounting=mounting)
    m = ROI_MASK.get(id(scan))
    if m is None or len(m) != len(p):
        return p
    return np.column_stack([p, m.astype(float)])

def find_block(points, min_height=None):
    if points is None or points.shape[1] < 4:
        return _orig_find_block(points, min_height)
    min_height = BLOCK_MIN_HEIGHT_MM if min_height is None else min_height
    inside = points[:, 3] == 1.0
    if inside.sum() < 50 or (~inside).sum() < 100:
        return None
    bed = float(np.nanpercentile(points[~inside, 2], 20))    # bed from outside the ROI
    roi = points[inside]
    top = roi[roi[:, 2] > bed + min_height]
    if len(top) < 50:
        return None
    lo = np.percentile(top[:, :2], 1, axis=0)
    hi = np.percentile(top[:, :2], 99, axis=0)
    return {"center": (lo + hi) / 2, "size": hi - lo, "lo": lo, "hi": hi,
            "bed_z": bed, "top_z": float(np.median(top[:, 2])), "n": int(len(top))}

picked = {}
for name, s in (("P1", P1), ("P2", P2), ("P3", P3)):
    if name in MANUAL:
        x0, y0, x1, y1 = MANUAL[name]
    elif len(CLICKS.get(name, [])) == 4:
        a = np.array(CLICKS[name]); x0, y0 = a.min(axis=0); x1, y1 = a.max(axis=0)
    else:
        raise RuntimeError(f"{name}: click 4 corners in the picking cell first (or give MANUAL['{name}'])")
    p = _orig_scan_points(s, mounting=lab.gocator.mounting)
    picked[name] = (float(x0), float(y0), float(x1), float(y1))
    ROI_MASK[id(s)] = ((p[:, 0] > x0 - MARGIN_MM) & (p[:, 0] < x1 + MARGIN_MM) &
                       (p[:, 1] > y0 - MARGIN_MM) & (p[:, 1] < y1 + MARGIN_MM))
    print(f"{name}: picked {x1 - x0:.0f} x {y1 - y0:.0f} mm "
          f"(block is {BLOCK_X_MM:.0f} x {BLOCK_Y_MM:.0f}); {ROI_MASK[id(s)].sum()} pts in ROI")

store.save_corners(picked)


In [ ]:
from scipy import ndimage
RING_MM = 80.0   # local bed = median Z of this ring just outside the ROI

def find_block(points, min_height=None, cell=2.0):
    if points is None or points.shape[1] < 4:
        return _orig_find_block(points, min_height)
    min_height = BLOCK_MIN_HEIGHT_MM if min_height is None else min_height
    inside = points[:, 3] == 1.0
    if inside.sum() < 50:
        return None
    lo_roi = points[inside, :2].min(axis=0); hi_roi = points[inside, :2].max(axis=0)
    near = np.all((points[:, :2] > lo_roi - RING_MM) & (points[:, :2] < hi_roi + RING_MM), axis=1)
    ring = near & ~inside
    if ring.sum() < 100:
        return None
    bed = float(np.nanmedian(points[ring, 2]))
    roi = points[inside]
    top = roi[roi[:, 2] > bed + min_height]
    if len(top) < 50:
        return None
    x0, y0 = top[:, 0].min(), top[:, 1].min()
    ix = ((top[:, 0] - x0) / cell).astype(int); iy = ((top[:, 1] - y0) / cell).astype(int)
    img = np.zeros((ix.max() + 1, iy.max() + 1), bool); img[ix, iy] = True
    img = ndimage.binary_opening(ndimage.binary_closing(img, iterations=2), iterations=2)
    lab_img, n = ndimage.label(img)
    if n == 0:
        return None
    big = 1 + int(np.argmax(ndimage.sum(img, lab_img, range(1, n + 1))))
    blob = top[lab_img[ix, iy] == big]
    if len(blob) < 50:
        return None
    lo = np.percentile(blob[:, :2], 0.5, axis=0)
    hi = np.percentile(blob[:, :2], 99.5, axis=0)
    return {"center": (lo + hi) / 2, "size": hi - lo, "lo": lo, "hi": hi,
            "bed_z": bed, "top_z": float(np.median(blob[:, 2])), "n": int(len(blob))}

In [ ]:
CONFIG_MOUNTING = lab.gocator.mounting
rows = []
for m in candidate_mountings(CONFIG_MOUNTING.to_dict()["scan_z"]):
    score, blocks, parts = score_mounting(m, P1, P2, P3)
    rows.append((score, m, blocks, parts))
rows.sort(key=lambda r: r[0])
for score, m, _, parts in rows:
    tag = "  <- configured" if m.to_dict() == CONFIG_MOUNTING.to_dict() else ""
    detail = ", ".join(f"{k}={v:+.1f}" for k, v in parts.items()) if parts else "block not found in every pass"
    print(f"{score:8.1f}  {m.to_dict()}  {detail}{tag}")

BEST_SCORE, BEST_MOUNTING, BEST_BLOCKS, BEST_PARTS = rows[0]
check("a mounting makes all three passes agree",
      BEST_PARTS and BEST_PARTS["size_err"] < 15 and abs(BEST_PARTS["dy12"]) < 5 and abs(BEST_PARTS["dx13"]) < 25,  # fwd/rev start-latency residual, not a mounting error
      str(BEST_PARTS))
check("the configured mounting is that one", BEST_MOUNTING.to_dict() == CONFIG_MOUNTING.to_dict(),
      f"best is {BEST_MOUNTING.to_dict()}")

b1, b2, b3 = BEST_BLOCKS
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, s, name, b in zip(axes, (P1, P2, P3), ("P1 fwd", "P2 fwd, stepped", "P3 reverse"), BEST_BLOCKS):
    if b is None:
        ax.set_title(f"{name}: no block"); continue
    p = scan_points(s, mounting=BEST_MOUNTING)
    near = np.all(np.abs(p[:, :2] - b["center"]) < [BLOCK_X_MM / 2 + 100, BLOCK_Y_MM / 2 + 100], axis=1)
    q = p[near][::max(1, near.sum() // 40000)]
    ax.scatter(q[:, 0], q[:, 1], c=q[:, 2], s=1)
    ax.add_patch(plt.Rectangle(b["lo"], *b["size"], fill=False, ec="r"))
    ax.set_title(f"{name}: block {b['size'][0]:.0f} x {b['size'][1]:.0f} mm")
    ax.set_xlabel("X (gantry - offset), mm"); ax.set_aspect("equal")
axes[0].set_ylabel("Y (gantry - offset), mm"); plt.tight_layout(); plt.show()

## 1f. Solve the Gocator translation; YAML to paste

`translation = B_W − Gocator block centre`, in X and Y. The Gocator centre along X averages the
forward (P1) and reverse (P3) passes, so any residual seam cancels out of the translation instead
of being baked into it.

Z is left at the configured value. An absolute height offset would need a WTT12L height
calibration, and none is assumed here.

In [ ]:
c_x = (b1["center"][0] + b3["center"][0]) / 2
c_y = (b1["center"][1] + b2["center"][1] + b3["center"][1]) / 3
old_t = lab.frames.frame_for("gocator").offset
NEW_T = np.array([B_W[0] - c_x, B_W[1] - c_y, old_t[2]])
print(f"old translation {np.round(old_t, 1).tolist()}  ->  new {np.round(NEW_T, 1).tolist()}")

trial = FrameRegistry(instruments={"gocator": InstrumentFrame("gocator", AffineTransform.from_translation(NEW_T))})
for name, s in (("P1", P1), ("P2", P2), ("P3", P3)):
    b = find_block(scan_points(s, frames=trial, mounting=BEST_MOUNTING))
    d = b["center"] - B_W
    print(f"{name}: block centre - B_W = ({d[0]:+.1f}, {d[1]:+.1f}) mm")

m = BEST_MOUNTING.to_dict()
print(f'''
# ---- paste into {CONFIG} ----
gocator:
  mounting:
    scan_x: {m["scan_x"]}
    scan_y: {m["scan_y"]}
    scan_z: {m["scan_z"]}

frames:
  instruments:
    gocator:
      translation: [{NEW_T[0]:.1f}, {NEW_T[1]:.1f}, {NEW_T[2]:.1f}]   # solved {time.strftime("%Y-%m-%d")} vs the WTT12L dot
''')
store.save_solution(B_W, m, NEW_T, transects=globals().get("wtt"))


## 1g. Reload from the edited config

Edit `CONFIG` with the YAML above and save it. Then run this cell. It disconnects, rebuilds the lab
from the file, and checks the loaded values against the solution, so Part 2 can't run on a stale
config. The rebuilt lab starts in `safe_mode` again.

In [ ]:
lab.gantry.set_safe_mode(True)
lab.safety_monitor.stop()
lab.disconnect_all()

lab = FlumeLab(CONFIG, simulate=SIMULATE)
lab.add("gantry")
lab.add("gocator")
check("reconnect", lab.connect_all())
lab.watch_for_safety()
lab.gocator.configure(mode="surface")
check("config mounting matches the solution", lab.gocator.mounting.to_dict() == BEST_MOUNTING.to_dict(),
      f"{lab.gocator.mounting.to_dict()}")
loaded_t = lab.frames.frame_for("gocator").offset
check("config translation matches the solution", np.allclose(loaded_t, NEW_T, atol=0.5),
      f"{np.round(loaded_t, 1).tolist()} vs {np.round(NEW_T, 1).tolist()}")

## 2a. Plan the tile from a region of interest (no motion)

Set `SURVEY_X_MM` (along travel) and `SURVEY_Y_MM` (across) in the parameters cell, in the
**experiment frame**, in mm. Leave either as `None` to centre it on the block. `Tile.from_roi()` reads
the swath from the connected Gocator's live active area and works out the rest:

- **Region wider than one swath:** the fewest passes that keep at least `MIN_OVERLAP` between
  neighbours, spread evenly so every seam has the same overlap and the passes cover the region exactly.
- **Region no wider than one swath:** a single pass with the swath centred on the region. That's a valid
  plan, but there is no seam in it, so the seam checks below will say so.

`SCAN_GANTRY_AXIS` fixes the **gantry** axis the passes scan along. `Tile.from_roi(..., gantry_axis=...)` works out which
experiment axis that is from the frame's rotation (the same letter at 0/180°, the other one at ±90°), so the region is always
given in experiment coordinates. §2c's seam analysis measures along experiment X, so it only applies when that is the travel axis.

Passes alternate direction (serpentine): pass 0 forward, pass 1 reverse, and so on.

The seam test measures the block where passes 0 and 1 overlap, so the cell checks the block sits in that
Y band (`pass 1's near edge` to `pass 0's far edge`) and is clear of the X ends of the survey where the
axis ramps. With more than two passes all of them run, and the analysis compares the first two.

Both runs use the same geometry:

- **Run A** — `accel_mm_s2=0.0`: no lead-in, the old 0.5 s settle, and the scan anchored to a live
  position read. This is the behaviour from issue #58.
- **Run B** — `accel_mm_s2=None`: `SurveyRunner` reads the accel from the axis, giving the ramp
  lead-in and the cruise-start anchor.

Every pass's gantry targets are checked against the soft limits and fences before anything moves.
If that check fails on Y, shrink `SURVEY_Y_MM`, narrow the active area's width in the config, or move
the block. `plot_survey_plan(TILE_B, lab=lab)` draws the plan.

In [ ]:
# Geometry (mm)
WTT_MARGIN_MM = 60.0        # WTT12L transects run this far past each block edge
ALIGN_PASS_LEN_MM = 600.0   # §1d pass length along X, centred on the block
ALIGN_STEP_Y_MM = None      # §1d P1 -> P2 step along Y; None = 0.4 x active-area width
# §2 survey boundaries, experiment frame, mm. None = centred on the block.
SURVEY_X_MM = (-900, 800)          # (x_min, x_max) along travel; None = TILE_LEN_MM centred on the block
SURVEY_Y_MM = (0, 1200)          # (y_min, y_max) across; passes + overlap are SOLVED to cover this; narrower than one swath = a single pass
# TILE_LEN_MM = 600.0         # used only when SURVEY_X_MM is None
MIN_OVERLAP = 0.10          # fewest passes that keep at least this swath overlap (spread evenly)
SCAN_GANTRY_AXIS = "X"     # gantry axis every pass scans along, whatever the experiment frame's rotation


In [ ]:
from laguna.robot.macron.fences import TrajectoryChecker

z_here = lab.frames.experiment_point_for("gocator", [START["X"], START["Y"], pos("Z")])[2]
b_exp = lab.frames.experiment_point_for("wtt12l", [B_W[0], B_W[1], pos("Z")])
AREA = lab.gocator.get_active_area()
SWATH = AREA["width_mm"]

# Survey region, experiment frame. Defaults centre it on the block (X: TILE_LEN_MM, Y: 1.7 swaths).
sx = tuple(SURVEY_X_MM) if SURVEY_X_MM else (b_exp[0] - TILE_LEN_MM / 2, b_exp[0] + TILE_LEN_MM / 2)
sy = tuple(SURVEY_Y_MM) if SURVEY_Y_MM else (b_exp[1] - 1.7 * SWATH / 2, b_exp[1] + 1.7 * SWATH / 2)

def make_tile(accel):
    # The swath comes from the lab's live active area; passes and overlap are solved to cover the region.
    return Tile.from_roi(sx, sy, z_here, lab=lab, min_overlap=MIN_OVERLAP, gantry_axis=SCAN_GANTRY_AXIS, serpentine=True,
                         scan_speed=SCAN_SPEED, travel_speed=TRAVEL_SPEED, accel_mm_s2=accel)

TILE_A, TILE_B = make_tile(0.0), make_tile(None)
TILE_PASSES = TILE_B.passes()
N_PASSES = len(TILE_PASSES)
print(f"survey X {sx[0]:.0f}..{sx[1]:.0f}  Y {sy[0]:.0f}..{sy[1]:.0f} (experiment, mm); swath {SWATH:.0f}")
if N_PASSES == 1:
    print(f"one pass covers it (region {sy[1] - sy[0]:.0f} mm wide <= swath {SWATH:.0f} mm), swath centred on the region")
else:
    print(f"{N_PASSES} passes, overlap {TILE_B.overlap:.1%}, pitch {TILE_B.pitch_mm:.0f} mm")
check("the passes cover the region", TILE_B.coverage_mm() >= (sy[1] - sy[0]) - 1e-6 or N_PASSES == 1,
      f"covering {TILE_B.coverage_mm():.0f} mm of {sy[1] - sy[0]:.0f} mm")
check("seam test needs two or more passes", N_PASSES >= 2,
      "a single pass has no seam: widen SURVEY_Y_MM past one swath, or this plan is for coverage only")
if N_PASSES >= 2:
    strip = (TILE_PASSES[1].start[1], TILE_PASSES[0].start[1] + SWATH)   # the Y band passes 0 and 1 both image
    print(f"overlap strip of passes 0 and 1: Y {strip[0]:.0f}..{strip[1]:.0f}" + (f"  ({N_PASSES} passes will run)" if N_PASSES > 2 else ""))
    check("block sits inside the overlap strip of passes 0 and 1",
          strip[0] <= b_exp[1] - BLOCK_Y_MM / 2 and b_exp[1] + BLOCK_Y_MM / 2 <= strip[1],
          f"block Y {b_exp[1] - BLOCK_Y_MM / 2:.0f}..{b_exp[1] + BLOCK_Y_MM / 2:.0f}")
check("block is well inside the survey along X (clear of the pass ends)",
      sx[0] + 100 <= b_exp[0] - BLOCK_X_MM / 2 and b_exp[0] + BLOCK_X_MM / 2 <= sx[1] - 100,
      f"block X {b_exp[0] - BLOCK_X_MM / 2:.0f}..{b_exp[0] + BLOCK_X_MM / 2:.0f}")

limits = {a["name"]: (a.get("soft_negative_limit_mm", -np.inf), a.get("soft_positive_limit_mm", np.inf))
          for a in lab.config.get("gantry")["axes"]}

def dry_run(tile, tag):
    # Gantry targets for every pass, computed exactly as SurveyRunner will command them
    # (same reference point resolution), checked against soft limits and fences.
    runner = SurveyRunner(lab, tile)
    runner._fill_tile_accel()
    ok = True
    for p in tile.passes():
        ref = runner._resolve_reference_point(p)
        a = lab.frames.gantry_target_for("gocator", list(p.start), reference_point=ref)
        b = lab.frames.gantry_target_for("gocator", list(p.end), reference_point=ref)
        inside = all(limits[n][0] <= v <= limits[n][1] for n, v in zip("XYZ", a)) and \
                 all(limits[n][0] <= v <= limits[n][1] for n, v in zip("XYZ", b))
        hits = lab.gantry.checker.check_ribbon(tuple(a), tuple(b))
        ok &= inside and not hits
        print(f"{tag} pass {p.index}: gantry {np.round(a, 1).tolist()} -> {np.round(b, 1).tolist()}"
              f"{'' if inside else '  OUTSIDE SOFT LIMITS'}{'  FENCE: ' + str(hits) if hits else ''}")
    return ok

print(TILE_B.describe())
plan_ok = (dry_run(TILE_A, "A") if RUN_BASELINE else True) & dry_run(TILE_B, "B")
check("every pass inside soft limits and clear of fences", plan_ok,
      "adjust SURVEY_X_MM / SURVEY_Y_MM, or move the block" if not plan_ok else "")

In [ ]:
from laguna.viz import plot_survey_plan
fig, ax = plot_survey_plan(TILE_B, lab=lab)      # with a connected lab (read-only)
# fig, ax = plot_survey_plan(TILE_B, lab.frames)   # planned geometry only

## 2b. Re-enable motion; run A, then run B

Same STOP cell as before; run it any time. The runs save every pass as `.npz` under `OUT`. The
configured `gocator.scan.formats` are also written to `gocator.output_dir`.

In [ ]:
# ===================== STOP CELL =====================
lab.pause()

In [ ]:
if not ALLOW_MOTION:
    raise SystemExit("ALLOW_MOTION is False — set it True in the parameters cell once the rig is clear.")
assert plan_ok, "§2a found a pass outside the soft limits or through a fence"
if lab.safety_state is not SafetyState.RUNNING:
    lab.resume()
check("set_safe_mode(False)", lab.gantry.set_safe_mode(False))

def run_tile(tile, tag):
    runner = SurveyRunner(lab, tile)
    done = runner.run(keep_results=True)
    for p, s in zip(done, runner.results):
        s.save_npz(OUT / f"run{tag}_pass{p.index}.npz")
    return done, runner.results

RUNS = {}
if RUN_BASELINE:
    RUNS["A"] = run_tile(TILE_A, "A")
RUNS["B"] = run_tile(TILE_B, "B")
print("accel used by run B:", TILE_B.accel_mm_s2, "mm/s²")

## 2c. Seam analysis (no motion)

For each run, both passes are oriented with the reloaded `lab.frames`, then compared:

- **Δtravel** = block centre in pass 1 (reverse) − pass 0 (forward), along the travel axis (experiment X or
  Y, whichever the passes run along; the analysis below follows it). This is the seam.
- **Texture Δtravel**: cross-correlation of the overlap strip along the travel axis. A second measure, which also
  uses the bed texture around the block. Treat it as inconclusive if its peak correlation is low.
- **ΔZ**: the median bed height difference in the strip. This is the separate tilt issue, reported
  only.
- **vs WTT12L**: each pass's block centre minus the WTT12L reference.

The prediction for run A comes from the issue's diagnosis. The old trigger fired 0.5 s into the
move, while the scan was anchored at the start, so Δtravel = 2 × the distance covered in that 0.5 s.
In run B, any residual most likely means the motion starts later than the trigger timer assumes,
by roughly δ = −Δtravel / (2v).

In [ ]:
v, a = SCAN_SPEED, TILE_B.accel_mm_s2
t_i = 0 if TILE_B.axis == "X" else 1          # experiment axis the passes travel along
s_i = 1 - t_i                                 # ... and the one they step across
T = "XY"[t_i]
SUMMARY = {"accel_mm_s2": a, "scan_speed": v, "B_W_gantry": B_W.tolist(), "block_experiment": b_exp[:2].tolist(),
           "travel_axis_experiment": T, "scan_axis_gantry": SCAN_GANTRY_AXIS}
fig, axes = plt.subplots(len(RUNS), 2, figsize=(13, 4 * len(RUNS)), squeeze=False)
for row, (tag, (done, scans)) in zip(axes, RUNS.items()):
    pts = [orient_scan(s, frames=lab.frames).to_points(drop_invalid=True) for s in scans]
    blocks = [find_block(p) for p in pts]
    if any(b is None for b in blocks):
        check(f"run {tag}: block found in both passes", False); continue
    tp = [p[:, [t_i, s_i, 2]] for p in pts]    # (travel, across, height): the analysis below is axis-agnostic
    c0, c1 = blocks[0]["center"], blocks[1]["center"]
    d_travel, d_across = c1[t_i] - c0[t_i], c1[s_i] - c0[s_i]
    slo = max(np.percentile(tp[0][:, 1], 1), np.percentile(tp[1][:, 1], 1))
    shi = min(np.percentile(tp[0][:, 1], 99), np.percentile(tp[1][:, 1], 99))
    tlo = max(tp[0][:, 0].min(), tp[1][:, 0].min()) + 20
    thi = min(tp[0][:, 0].max(), tp[1][:, 0].max()) - 20
    strip = [p[(p[:, 1] > slo) & (p[:, 1] < shi)] for p in tp]
    shift, corr = travel_shift(strip[0], strip[1], (slo, shi), (tlo, thi))
    dz = float(np.median(strip[1][:, 2]) - np.median(strip[0][:, 2]))
    vs_wtt = [(b["center"] - b_exp[:2]).round(1).tolist() for b in blocks]
    rec = {"d_travel_mm": float(d_travel), "d_across_mm": float(d_across), "texture_d_travel_mm": shift,
           "texture_corr": corr, "bed_dZ": dz, "pass_minus_wtt12l": vs_wtt, "passes": [p.to_dict() for p in done]}
    if tag == "A":
        rec["predicted_d_travel_mm"] = 2 * travel_at(0.5, v, a)
    else:
        rec["implied_latency_s"] = float(-d_travel / (2 * v))
    SUMMARY[f"run_{tag}"] = rec
    print(f"run {tag}: d_travel({T})={d_travel:+.1f} mm  d_across={d_across:+.1f} mm  texture d_travel={shift:+.1f} mm "
          f"(corr {corr:.2f})  bed dZ={dz:+.1f} mm  vs WTT12L {vs_wtt}"
          + (f"  predicted d_travel={rec['predicted_d_travel_mm']:+.1f} mm" if tag == "A" else
             f"  implied start latency {rec['implied_latency_s'] * 1000:+.0f} ms"))

    ax = row[0]
    near = np.all(np.abs(pts[0][:, :2] - c0) < blocks[0]["size"] / 2 + 100, axis=1)
    q = pts[0][near][::max(1, near.sum() // 40000)]
    ax.scatter(q[:, 0], q[:, 1], c=q[:, 2], s=1, cmap="Greys")
    for b, color, ls, name in zip(blocks, ("C0", "C1"), ("-", "--"), ("pass 0 fwd", "pass 1 rev")):
        ax.add_patch(plt.Rectangle(b["lo"], *b["size"], fill=False, ec=color, ls=ls, lw=2, label=name))
    ax.set_title(f"run {tag}: block as each pass sees it"); ax.set_aspect("equal")
    ax.set_xlabel("experiment X, mm"); ax.set_ylabel("experiment Y, mm"); ax.legend(loc="upper right")
    ax = row[1]
    for p, b, name in zip(tp, blocks, ("pass 0 fwd", "pass 1 rev")):
        line = p[np.abs(p[:, 1] - b["center"][s_i]) < 10]
        line = line[np.abs(line[:, 0] - b["center"][t_i]) < b["size"][t_i]]
        ax.plot(line[:, 0], line[:, 2], ".", ms=1, label=name)
    ax.set_title(f"run {tag}: {T} profile through the block (d_travel = {d_travel:+.1f} mm)")
    ax.set_xlabel(f"experiment {T} (travel), mm"); ax.legend()
plt.tight_layout(); plt.show()

if "run_B" in SUMMARY:
    check("run B: forward and reverse agree along travel", abs(SUMMARY["run_B"]["d_travel_mm"]) <= SEAM_TOL_MM,
          f"d_travel = {SUMMARY['run_B']['d_travel_mm']:+.1f} mm, tolerance {SEAM_TOL_MM}")
    check("run B: block lands on the WTT12L reference",
          all(abs(c) < 3 * SEAM_TOL_MM for d in SUMMARY["run_B"]["pass_minus_wtt12l"] for c in d),
          str(SUMMARY["run_B"]["pass_minus_wtt12l"]))
if "run_A" in SUMMARY:
    check("run A: the old seam is visible (the test can detect it)",
          abs(SUMMARY["run_A"]["d_travel_mm"]) > 3 * SEAM_TOL_MM, f"d_travel = {SUMMARY['run_A']['d_travel_mm']:+.1f} mm")


## 2d. Summary, park, safe_mode back on

In [ ]:
try:
    lab.move_to(X=START["X"], Y=START["Y"], speed=TRAVEL_SPEED).wait()
except Exception as exc:
    print("could not return to start:", exc)
lab.gantry.set_safe_mode(True)
SUMMARY["checks"] = [{"name": n, "ok": ok, "detail": d} for n, ok, d in results]
json.dump(SUMMARY, open(OUT / "summary.json", "w"), indent=2, default=float)
print(f"\n{sum(ok for _, ok, _ in results)}/{len(results)} checks passed — summary in {OUT / 'summary.json'}")
for name, ok, detail in results:
    if not ok:
        print("FAIL", name, "—", detail)

In [ ]:
lab.safety_monitor.stop()
lab.disconnect_all()